In [1]:
import os

def print_directory_tree(startpath, max_depth=None, exclude_hidden=True):
    """
    以树状图形式打印文件结构。
    
    :param startpath: 要查看的根目录路径
    :param max_depth: 最大读取层级 (None 表示读取到底)
    :param exclude_hidden: 是否排除隐藏文件/文件夹 (以 '.' 开头的)
    """
    print(f"📦 [{os.path.abspath(startpath)}]")
    
    for root, dirs, files in os.walk(startpath):
        # 计算当前层级深度
        level = root.replace(startpath, '').count(os.sep)
        
        # 深度控制：如果超过最大深度，清空 dirs 以停止向下遍历
        if max_depth is not None and level >= max_depth:
            dirs.clear()
            continue
            
        # 排除隐藏文件夹（如 .git, .ipynb_checkpoints）
        if exclude_hidden:
            dirs[:] = [d for d in dirs if not d.startswith('.')]
            
        # 打印当前文件夹
        indent = '│   ' * level
        if level > 0:
            print(f"{indent}├── 📂 {os.path.basename(root)}/")
        
        # 打印当前文件夹下的文件
        subindent = '│   ' * (level + 1)
        
        # 过滤隐藏文件
        if exclude_hidden:
            files = [f for f in files if not f.startswith('.')]
            
        for i, f in enumerate(files):
            # 判断是否是该目录下的最后一个文件，调整前缀符号
            if i == len(files) - 1 and not dirs:
                print(f"{subindent[:-4]}└── 📄 {f}")
            else:
                print(f"{subindent[:-4]}├── 📄 {f}")

# ================= 测试运行 =================
if __name__ == "__main__":
    # 查看当前目录，最多查看 2 层深度
    target_dir = "." 
    print_directory_tree(target_dir, max_depth=3)

📦 [d:\desktop\补充实验\HiPro-LoRA\imageCodeAndResult]
├── 📄 create_Image.ipynb
├── 📄 llm-result.md
├── 📄 sensitivity_add_result.md
├── 📄 Times_New_Roman.ttf
│   ├── 📂 smp2020_add_sensitivity/
│   ├── 📄 smp2020_ours_final.csv
│   └── 📄 smp2020_sensitivity_add.csv
│   ├── 📂 smp2020_fine_grained_ablation/
│   ├── 📄 smp2020_fine_grained_ablation.csv
│   └── 📄 smp2020_fine_grained_ablation_Summary.md
│   ├── 📂 smp2020_main/
│   ├── 📄 smp2020_main.csv
│   ├── 📄 smp2020_main_Summary.md
│   └── 📄 smp2020_sensitivity.csv
│   ├── 📂 sst5_add_sensitivity/
│   ├── 📄 sst5_ours_final.csv
│   └── 📄 sst5_sensitivity_add.csv
│   ├── 📂 sst5_fine_grained_ablation/
│   ├── 📄 sst5_fine_grained_ablation.csv
│   └── 📄 sst5_fine_grained_ablation_Summary.md
│   ├── 📂 sst5_main/
│   ├── 📄 sst5_main.csv
│   ├── 📄 sst5_main_Summary.md
│   └── 📄 sst5_sensitivity.csv
│   ├── 📂 tweeteval_add_sensitivity/
│   ├── 📄 tweeteval_ours_final.csv
│   └── 📄 tweeteval_sensitivity_add.csv
│   ├── 📂 tweeteval_fine_grained_ablation/


In [3]:
import os
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.font_manager as font_manager
import seaborn as sns
import warnings
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

# ==================== Font ====================
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()
font_path = os.path.join(BASE_DIR, 'Times_New_Roman.ttf')
if os.path.exists(font_path):
    font_manager.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'Times New Roman'
else:
    plt.rcParams['font.family'] = 'serif'

# ==================== Output ====================
FIG_DIR = os.path.join(BASE_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# ==================== Global Style ====================
plt.rcParams.update({
    'font.size': 36,
    'axes.titlesize': 64,
    'axes.labelsize': 52,
    'xtick.labelsize': 48,
    'ytick.labelsize': 48,
    'legend.fontsize': 32,
    'figure.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 2.0,
    'lines.linewidth': 5.0,
    'lines.markersize': 14,
    'grid.linewidth': 1.5,
    'mathtext.fontset': 'stix',
    'pdf.fonttype': 42
})

# ==================== Colors ====================
C_OURS       = '#F05A28'
C_BLUE_MAIN  = '#1E5FA0'
C_BLUE_LIGHT = '#9CC4E4'
C_GREY_MAIN  = '#7F8C8D'
C_GREY_LIGHT = '#D3D3D3'
C_ADV        = '#2E8B57'
C_DORA       = '#27AE60'
C_LORA_BAL   = '#8E44AD'

HATCH_NO_COT = '//'
HATCH_COT    = '\\\\'
HATCH_OURS   = None

METHOD_COLOR_MAP = {
    'LoRA-Ours':            C_OURS,
    'LoRA-Adv':             C_ADV,
    'DoRA-Balanced':        C_DORA,
    'Full-FineTuning':      C_GREY_MAIN,
    'LoRA-Balanced':        C_LORA_BAL,
    'LoRA-Ablation-NoMem':  C_BLUE_LIGHT,
    'LoRA-Ablation-NoHSP':  C_BLUE_MAIN
}

METHOD_NAME_MAP = {
    'LoRA-Ours':            'HiPro-LoRA',
    'LoRA-Adv':             'LoRA-Adv',
    'DoRA-Balanced':        'DoRA',
    'Full-FineTuning':      'Full-FT',
    'LoRA-Balanced':        'LoRA-Bal',
    'LoRA-Ablation-NoMem':  'w/o TPMB',
    'LoRA-Ablation-NoHSP':  'w/o AHSP'
}

MARKER_MAP = {
    'LoRA-Ours':        's',
    'LoRA-Adv':         '^',
    'DoRA-Balanced':    '*',
    'LoRA-Balanced':    'o',
    'Full-FineTuning':  'X'
}

# ==================== Selected Params (Sensitivity) ====================
SELECTED_PARAMS = {
    "SMP2020":   {"MemSize": 2000, "TailWeight": 3.0, "Temperature": 0.1,  "LossWeight": 0.1},
    "SST-5":     {"MemSize": 1200, "TailWeight": 1.0, "Temperature": 0.5,  "LossWeight": 0.2},
    "TweetEval": {"MemSize": 1500, "TailWeight": 2.0, "Temperature": 0.2,  "LossWeight": 0.06}
}

# ==================== Data Loaders ====================
def load_main_data(filepath):
    if not os.path.exists(filepath):
        return pd.DataFrame()
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    sep_idx = next((i for i, l in enumerate(lines) if '|' in l and '---' in l), -1)
    if sep_idx == -1:
        return pd.DataFrame()
    headers = [h.strip() for h in lines[sep_idx - 1].strip().strip('|').split('|')]
    rows = []
    for line in lines[sep_idx + 1:]:
        line = line.strip()
        if not line or not line.startswith('|'):
            continue
        cells = [c.strip() for c in line.strip('|').split('|')]
        rows.append(cells[:len(headers)])
    df = pd.DataFrame(rows, columns=headers)
    for col in df.columns:
        if col == 'N' or any(k in col for k in ['Mean', 'Accuracy', 'F1', 'AUC', 'Time', 'Memory']):
            df[col] = df[col].apply(
                lambda x: float(str(x).split('±')[0].strip()) if '±' in str(x) else pd.to_numeric(x, errors='coerce')
            )
    return df


def load_sensitivity_data(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    data_lines = [l for l in lines if l.strip().startswith('|') and '---' not in l]
    md_content = '\n'.join(data_lines)
    df = pd.read_csv(io.StringIO(md_content), sep='|')
    df = df.dropna(axis=1, how='all')
    df.columns = df.columns.str.replace('*', '').str.strip()
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].str.replace('*', '').str.strip()
    df['Value'] = pd.to_numeric(df['Value'])
    seed_cols = [c for c in df.columns if 'Seed' in c]
    df_melted = df.melt(
        id_vars=['Dataset', 'Type', 'Value'],
        value_vars=seed_cols,
        var_name='Seed',
        value_name='Macro_F1'
    )
    df_melted['Macro_F1'] = pd.to_numeric(df_melted['Macro_F1']) * 100
    return df_melted


# ==================== Figure: LLM Comparison ====================
def plot_llm_comparison(df_smp, df_sst, df_tweet):
    llm_path = os.path.join(BASE_DIR, 'llm-result.md')
    if not os.path.exists(llm_path):
        print(f"File not found: {llm_path}")
        return

    with open(llm_path, 'r', encoding='utf-8') as f:
        raw_lines = f.readlines()

    table_data = []
    for line in raw_lines:
        clean_line = line.strip().replace('*', '')
        if '|' in clean_line and '---' not in clean_line:
            cells = [c.strip() for c in clean_line.split('|')]
            if cells and cells[0] == '':
                cells = cells[1:]
            if cells and cells[-1] == '':
                cells = cells[:-1]
            if cells:
                table_data.append(cells)

    if len(table_data) < 2:
        return

    df_llm = pd.DataFrame(table_data[1:], columns=table_data[0])
    col_ds   = 'Dataset'
    col_md   = 'Model'
    col_meth = 'Method'
    col_f1   = 'Macro-F1'
    df_llm[col_ds] = df_llm[col_ds].replace(r'^\s*$', np.nan, regex=True).ffill()

    datasets = ['SMP2020', 'SST-5', 'TweetEval']
    data_qwen  = {'0-nocot': [], '1-nocot': [], '0-cot': [], '1-cot': []}
    data_llama = {'0-nocot': [], '1-nocot': [], '0-cot': [], '1-cot': []}
    data_ours  = []

    ours_sources = {
        'SMP2020':   (df_smp,   1000),
        'SST-5':     (df_sst,   1150),
        'TweetEval': (df_tweet, 1000)
    }

    for ds in datasets:
        def get_f1(model_kw, method_str):
            row = df_llm[
                (df_llm[col_ds].str.contains(ds)) &
                (df_llm[col_md].str.contains(model_kw)) &
                (df_llm[col_meth].str.strip() == method_str)
            ]
            return float(row[col_f1].iloc[0]) * 100 if not row.empty else 0

        data_qwen['0-nocot'].append(get_f1('Qwen', 'Zero-Shot (No CoT)'))
        data_qwen['1-nocot'].append(get_f1('Qwen', 'Balanced 1-Shot (No CoT)'))
        data_qwen['0-cot'].append(get_f1('Qwen', '0-Shot CoT'))
        data_qwen['1-cot'].append(get_f1('Qwen', 'Balanced 1-Shot CoT'))

        data_llama['0-nocot'].append(get_f1('Llama', 'Zero-Shot (No CoT)'))
        data_llama['1-nocot'].append(get_f1('Llama', 'Balanced 1-Shot (No CoT)'))
        data_llama['0-cot'].append(get_f1('Llama', '0-Shot CoT'))
        data_llama['1-cot'].append(get_f1('Llama', 'Balanced 1-Shot CoT'))

        df_src, n_val = ours_sources[ds]
        o_row = df_src[(df_src['N'] == n_val) & (df_src['Method'] == 'LoRA-Ours')]
        data_ours.append(o_row['Macro-F1 (Mean±Std)'].iloc[0] * 100 if not o_row.empty else 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(28, 12))
    x     = np.arange(len(datasets))
    width = 0.16

    for ax, data_llm_model, title in [
        (ax1, data_qwen,  '(A) Qwen-2.5-7B'),
        (ax2, data_llama, '(B) Llama-3.1-8B')
    ]:
        ax.bar(x - 2*width, data_llm_model['0-nocot'], width, color=C_GREY_LIGHT, edgecolor='black', linewidth=1.5, hatch=HATCH_NO_COT)
        ax.bar(x - width,   data_llm_model['1-nocot'], width, color=C_GREY_MAIN,  edgecolor='black', linewidth=1.5, hatch=HATCH_NO_COT)
        ax.bar(x,           data_llm_model['0-cot'],   width, color=C_BLUE_LIGHT, edgecolor='black', linewidth=1.5, hatch=HATCH_COT)
        ax.bar(x + width,   data_llm_model['1-cot'],   width, color=C_BLUE_MAIN,  edgecolor='black', linewidth=1.5, hatch=HATCH_COT)
        ax.bar(x + 2*width, data_ours,                 width, color=C_OURS,       edgecolor='black', linewidth=1.5, zorder=3)
        ax.set_ylabel('Macro-F1 Score (%)', fontweight='bold')
        ax.set_title(title, fontweight='bold', pad=25)
        ax.set_xticks(x)
        ax.set_xticklabels(datasets, fontweight='bold')
        ax.tick_params(axis='x', pad=20)
        ax.set_ylim(0, 85)
        ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=2)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        for container in ax.containers:
            ax.bar_label(container, fmt='%.1f', padding=8, fontweight='bold', fontsize=26)

    legend_patches = [
        Patch(facecolor=C_GREY_LIGHT, edgecolor='black', hatch=HATCH_NO_COT, label='Zero-Shot (No CoT)'),
        Patch(facecolor=C_GREY_MAIN,  edgecolor='black', hatch=HATCH_NO_COT, label='1-Shot Bal (No CoT)'),
        Patch(facecolor=C_BLUE_LIGHT, edgecolor='black', hatch=HATCH_COT,    label='Zero-Shot (CoT)'),
        Patch(facecolor=C_BLUE_MAIN,  edgecolor='black', hatch=HATCH_COT,    label='1-Shot Bal (CoT)'),
        Patch(facecolor=C_OURS,       edgecolor='black', hatch=HATCH_OURS,   label='HiPro-LoRA (Ours)')
    ]
    fig.legend(handles=legend_patches, loc='lower center', bbox_to_anchor=(0.5, -0.15),
               ncol=3, frameon=False, fontsize=42, columnspacing=2.0)
    plt.tight_layout(rect=[0, 0, 1, 1])
    plt.savefig(os.path.join(FIG_DIR, 'figure4_llm.pdf'))
    plt.close()
    print("figure4_llm.pdf saved.")


# ==================== Figure: Efficiency ====================
def plot_efficiency(df_smp, df_sst, df_tweet):
    tasks      = [('SMP2020', df_smp, 1000), ('TweetEval', df_tweet, 1000), ('SST-5', df_sst, 1150)]
    main_m     = ['LoRA-Ours', 'LoRA-Adv', 'DoRA-Balanced', 'LoRA-Balanced', 'Full-FineTuning']
    fig, axes  = plt.subplots(1, 3, figsize=(32, 15))
    legend_handles = []
    method_seen    = set()

    for i, (name, df, n_val) in enumerate(tasks):
        ax = axes[i]
        if df.empty:
            continue
        subset = df[(df['N'] == n_val) & (df['Method'].isin(main_m))].copy()
        for m in main_m:
            row = subset[subset['Method'] == m]
            if row.empty:
                continue
            row = row.iloc[0]
            t   = row['Train_Time_Sec (Mean±Std)']
            mem = row['Peak_Memory_MB (Mean±Std)']
            f1  = row['Macro-F1 (Mean±Std)']
            if pd.isna(t) or pd.isna(mem):
                continue
            col    = METHOD_COLOR_MAP.get(m, '#333333')
            marker = MARKER_MAP.get(m, 'o')
            ax.scatter(t, mem, s=(f1 - 0.3) * 12000, color=col, alpha=0.7,
                       edgecolor='black', linewidth=2, marker=marker)
            if i == 0 and m not in method_seen:
                legend_handles.append(
                    mlines.Line2D([], [], color=col, marker=marker, linestyle='None',
                                  markersize=60, markeredgecolor='black', markeredgewidth=3,
                                  label=METHOD_NAME_MAP.get(m, m))
                )
                method_seen.add(m)

        ax.tick_params(axis='both', which='major', labelsize=52, width=3, length=10)
        ax.set_xlabel('Training Time (s)', fontweight='bold', fontsize=55)
        if i == 0:
            ax.set_ylabel('Peak Memory (MB)', fontweight='bold', fontsize=48)
        ax.set_title(f'({chr(65 + i)}) {name}', weight='bold', pad=30, fontsize=64)
        ax.grid(True, ls='--', alpha=0.3, linewidth=2)
        for spine in ax.spines.values():
            spine.set_linewidth(2.5)
        ax.set_xlim(subset['Train_Time_Sec (Mean±Std)'].min() * 0.7,
                    subset['Train_Time_Sec (Mean±Std)'].max() * 1.3)
        ax.set_ylim(subset['Peak_Memory_MB (Mean±Std)'].min() * 0.8,
                    subset['Peak_Memory_MB (Mean±Std)'].max() * 1.3)

    fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.02),
               ncol=5, frameon=False, prop={'size': 50})
    plt.tight_layout(rect=[0, 0.15, 1, 1])
    plt.savefig(os.path.join(FIG_DIR, 'figure5_efficiency.pdf'))
    plt.close()
    print("figure5_efficiency.pdf saved.")


# ==================== Figure: Sensitivity ====================
def plot_sensitivity():
    sens_path = os.path.join(BASE_DIR, 'sensitivity_add_result.md')
    full_df   = load_sensitivity_data(sens_path)

    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    axes      = axes.flatten()

    plot_configs = [
        {"type": "MemSize",     "ax_idx": 0, "title": "(a) Effect of Memory Size ($M$)",      "xlabel": "Memory Size"},
        {"type": "TailWeight",  "ax_idx": 1, "title": "(b) Effect of Tail Weight ($\\gamma$)", "xlabel": "Tail Weight"},
        {"type": "Temperature", "ax_idx": 2, "title": "(c) Effect of Temperature ($\\tau$)",   "xlabel": "Temperature"},
        {"type": "LossWeight",  "ax_idx": 3, "title": "(d) Effect of Loss Weight ($\\beta$)",  "xlabel": "Loss Weight"}
    ]

    datasets = full_df['Dataset'].unique()
    palette  = sns.color_palette("Set1", len(datasets))

    for cfg in plot_configs:
        ax     = axes[cfg["ax_idx"]]
        subset = full_df[full_df['Type'] == cfg["type"]]

        sns.lineplot(data=subset, x="Value", y="Macro_F1", hue="Dataset",
                     style="Dataset", markers=True, ax=ax,
                     palette=palette, errorbar='sd', legend=False)

        for i, ds in enumerate(datasets):
            target_val = SELECTED_PARAMS.get(ds, {}).get(cfg["type"])
            if target_val is not None:
                ds_data   = subset[subset['Dataset'] == ds]
                mean_line = ds_data.groupby('Value')['Macro_F1'].mean().sort_index()
                if not mean_line.empty:
                    interp_y = np.interp(target_val, mean_line.index, mean_line.values)
                    ax.scatter(target_val, interp_y,
                               marker='*', s=600,
                               color='gold', edgecolor='black',
                               linewidth=1.5, zorder=10)

        ax.set_title(cfg["title"], fontweight='bold', pad=25, fontsize=32)
        ax.set_xlabel(cfg["xlabel"], fontweight='bold', labelpad=15, fontsize=28)
        ax.tick_params(axis='both', labelsize=24)
        ax.grid(True, linestyle='--', alpha=0.5, linewidth=2.0)
        ax.margins(y=0.15)
        if cfg["ax_idx"] % 2 == 0:
            ax.set_ylabel("Macro-F1 (%)", fontweight='bold', labelpad=15, fontsize=28)
        else:
            ax.set_ylabel("")

    plt.subplots_adjust(hspace=0.4, wspace=0.15, bottom=0.15)

    from matplotlib.lines import Line2D
    custom_handles = []
    custom_labels  = []
    for i, ds in enumerate(datasets):
        custom_handles.append(
            Line2D([0], [0], color=palette[i], linestyle='-', linewidth=8, marker='o', markersize=20)
        )
        custom_labels.append(ds)
    custom_handles.append(
        Line2D([0], [0], color='w', marker='*', markerfacecolor='gold',
               markeredgecolor='black', markersize=36, markeredgewidth=2.0)
    )
    custom_labels.append("Selected Config")

    fig.legend(custom_handles, custom_labels,
               loc='upper center', bbox_to_anchor=(0.5, 0.06),
               ncol=len(custom_handles), frameon=False,
               fontsize=36, columnspacing=3.0, handletextpad=1.0)

    plt.savefig(os.path.join(FIG_DIR, 'figure6_sens.pdf'), bbox_inches='tight')
    plt.close()
    print("figure6_sens.pdf saved.")


# ==================== Main ====================
if __name__ == "__main__":
    df_smp   = load_main_data(os.path.join(BASE_DIR, 'smp2020_main',   'smp2020_main_results_Summary.md'))
    df_sst   = load_main_data(os.path.join(BASE_DIR, 'sst5_main',      'sst5_main_results_Summary.md'))
    df_tweet = load_main_data(os.path.join(BASE_DIR, 'tweeteval_main', 'tweeteval_main_results_Summary.md'))

    plot_llm_comparison(df_smp, df_sst, df_tweet)
    plot_efficiency(df_smp, df_sst, df_tweet)
    plot_sensitivity()

figure4_llm.pdf saved.
figure5_efficiency.pdf saved.
figure6_sens.pdf saved.
